# Why do VLMs misread analog clocks? -- Step 1: Behavior Check

This notebook is a **behavior check only** -- no interpretability yet. It:

1. Generates 500 synthetic analog clock images with known ground-truth times.
2. Asks **Qwen2.5-VL-3B-Instruct** to read each clock (`"What time does this clock show? Answer only in HH:MM format."`).
3. Analyzes the answers: overall accuracy, hour vs. minute accuracy, the **hand-swap rate**, accuracy within +/-5 minutes, and breakdowns by hour/minute.

Designed to run top-to-bottom on a free Kaggle notebook with a **T4 GPU (16GB)**. Turn on the GPU accelerator under *Settings > Accelerator* before running.

The three `%%writefile` cells below write out `clocks.py`, `eval_behavior.py`, and `analyze.py` -- these are the exact same scripts included in the project repo, just embedded here so the notebook is self-contained.

## 0. Setup: install packages

This must run first. Qwen2.5-VL needs a recent `transformers` version, and Kaggle's
preinstalled one is often too old to have the `Qwen2_5_VLForConditionalGeneration` class.

In [ ]:
# Kaggle already ships torch. Upgrade transformers (Qwen2.5-VL needs a recent
# version) and add the other packages this notebook needs.
!pip install -U transformers accelerate qwen-vl-utils


## 1. Clock generator (`clocks.py`)

Draws analog clocks with matplotlib: controllable hour/minute, hand lengths, hand thickness, numbers on/off, tick marks on/off. Default style is a clear white face with black hands, hour hand shorter and thicker than the minute hand.

In [ ]:
%%writefile clocks.py
"""
clocks.py -- Draw synthetic analog clock images with matplotlib.

This is STEP 1 of the clock-reading behavior check: we need a controllable
source of clock images with known ground truth (the true hour/minute) so we
can later check whether a vision-language model reads them correctly.

Usage as a script:
    python clocks.py --n 500 --out_dir data --seed 42

Usage as a library (e.g. from the notebook):
    from clocks import generate_dataset
    df = generate_dataset(n=500, out_dir="data", seed=42)
"""

import argparse
import math
import os

import matplotlib
matplotlib.use("Agg")  # no display needed, just save PNG files
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np


# ---------------------------------------------------------------------------
# Geometry helpers
# ---------------------------------------------------------------------------

def _polar_to_xy(angle_deg_from_12, length):
    """Convert an angle measured clockwise from the 12 o'clock position
    (i.e. how a clock hand angle is normally described) into (x, y)
    coordinates on a unit circle, with (0, 0) at the clock's center and
    (0, 1) being the 12 o'clock position.
    """
    # Standard math angles go counter-clockwise from the +x axis, so we
    # convert: clockwise-from-top angle theta -> math angle (90 - theta).
    theta = math.radians(90 - angle_deg_from_12)
    x = length * math.cos(theta)
    y = length * math.sin(theta)
    return x, y


def hand_angles(hour, minute):
    """Return (hour_hand_angle_deg, minute_hand_angle_deg), both measured
    clockwise from 12 o'clock.

    Public (not prefixed with `_`) so analyze.py can reuse the exact same
    geometry when checking whether the two hands nearly overlap.
    """
    # Minute hand: 360 degrees / 60 minutes = 6 degrees per minute.
    minute_angle = minute * 6.0

    # Hour hand: 360 degrees / 12 hours = 30 degrees per hour, PLUS it
    # creeps forward smoothly as minutes pass (0.5 degrees per minute).
    # Using hour % 12 so that hour=12 behaves like hour=0 (points to top).
    hour_angle = (hour % 12) * 30.0 + minute * 0.5

    return hour_angle, minute_angle


# ---------------------------------------------------------------------------
# Drawing a single clock
# ---------------------------------------------------------------------------

def generate_clock_image(
    hour,
    minute,
    save_path,
    hour_hand_length=0.50,
    minute_hand_length=0.85,
    hour_hand_thickness=9,
    minute_hand_thickness=4,
    show_numbers=True,
    show_ticks=True,
    face_color="white",
    hand_color="black",
    edge_color="black",
    image_size=512,
    dpi=100,
):
    """Draw one analog clock face showing `hour`:`minute` and save it as a PNG.

    Default style (per spec): clear white face, black hands, hour hand
    shorter AND thicker than the minute hand.

    Parameters
    ----------
    hour : int (1-12)
    minute : int (0-59)
    save_path : str, where to write the PNG
    hour_hand_length, minute_hand_length : float, hand length as a fraction
        of the clock face radius (radius = 1.0)
    hour_hand_thickness, minute_hand_thickness : float, matplotlib linewidth
    show_numbers : bool, draw the 1-12 numerals on the face
    show_ticks : bool, draw the 60 minute tick marks (with longer/thicker
        ticks at each hour)
    face_color, hand_color, edge_color : matplotlib color strings
    image_size : int, output image size in pixels (square)
    dpi : int, resolution used to convert the figure to pixels
    """
    figsize = image_size / dpi
    fig, ax = plt.subplots(figsize=(figsize, figsize), dpi=dpi)

    # --- clock face ---
    face = plt.Circle((0, 0), 1.0, facecolor=face_color, edgecolor=edge_color,
                       linewidth=2.5, zorder=1)
    ax.add_patch(face)

    # --- tick marks (60 of them; every 5th tick, i.e. each hour, is longer/thicker) ---
    if show_ticks:
        for m in range(60):
            angle = m * 6.0
            is_hour_tick = (m % 5 == 0)
            outer = 0.95
            inner = 0.80 if is_hour_tick else 0.88
            lw = 2.5 if is_hour_tick else 1.0
            x1, y1 = _polar_to_xy(angle, inner)
            x2, y2 = _polar_to_xy(angle, outer)
            ax.plot([x1, x2], [y1, y2], color=edge_color, linewidth=lw,
                     solid_capstyle="round", zorder=2)

    # --- numbers 1-12 ---
    if show_numbers:
        for h in range(1, 13):
            angle = h * 30.0
            x, y = _polar_to_xy(angle, 0.66)
            ax.text(x, y, str(h), ha="center", va="center",
                    fontsize=17, fontweight="bold", color=edge_color, zorder=2)

    # --- hands ---
    hour_angle, minute_angle = hand_angles(hour, minute)
    hx, hy = _polar_to_xy(hour_angle, hour_hand_length)
    mx, my = _polar_to_xy(minute_angle, minute_hand_length)

    # Hour hand: shorter and thicker (drawn first, minute hand on top)
    ax.plot([0, hx], [0, hy], color=hand_color, linewidth=hour_hand_thickness,
             solid_capstyle="round", zorder=3)
    # Minute hand: longer and thinner
    ax.plot([0, mx], [0, my], color=hand_color, linewidth=minute_hand_thickness,
             solid_capstyle="round", zorder=4)

    # center pivot dot
    ax.add_patch(plt.Circle((0, 0), 0.03, facecolor=hand_color, zorder=5))

    ax.set_xlim(-1.12, 1.12)
    ax.set_ylim(-1.12, 1.12)
    ax.set_aspect("equal")
    ax.axis("off")

    fig.savefig(save_path, dpi=dpi, bbox_inches=None, pad_inches=0)
    plt.close(fig)


# ---------------------------------------------------------------------------
# Dataset generation
# ---------------------------------------------------------------------------

def generate_dataset(
    n=500,
    out_dir="data",
    seed=42,
    hour_hand_length=0.50,
    minute_hand_length=0.85,
    hour_hand_thickness=9,
    minute_hand_thickness=4,
    show_numbers=True,
    show_ticks=True,
    face_color="white",
    hand_color="black",
    image_size=512,
):
    """Generate `n` clock images with random times (default style) and a CSV
    logging the true time + all rendering settings for each image.

    Returns the metadata DataFrame (also written to <out_dir>/data.csv).
    """
    os.makedirs(out_dir, exist_ok=True)
    rng = np.random.RandomState(seed)

    rows = []
    for i in range(n):
        hour = int(rng.randint(1, 13))     # 1-12 inclusive
        minute = int(rng.randint(0, 60))   # 0-59 inclusive
        filename = f"clock_{i:04d}.png"
        save_path = os.path.join(out_dir, filename)

        generate_clock_image(
            hour, minute, save_path,
            hour_hand_length=hour_hand_length,
            minute_hand_length=minute_hand_length,
            hour_hand_thickness=hour_hand_thickness,
            minute_hand_thickness=minute_hand_thickness,
            show_numbers=show_numbers,
            show_ticks=show_ticks,
            face_color=face_color,
            hand_color=hand_color,
            image_size=image_size,
        )

        rows.append({
            "filename": filename,
            "hour": hour,
            "minute": minute,
            "hour_hand_length": hour_hand_length,
            "minute_hand_length": minute_hand_length,
            "hour_hand_thickness": hour_hand_thickness,
            "minute_hand_thickness": minute_hand_thickness,
            "show_numbers": show_numbers,
            "show_ticks": show_ticks,
            "face_color": face_color,
            "hand_color": hand_color,
            "image_size": image_size,
        })

    df = pd.DataFrame(rows)
    csv_path = os.path.join(out_dir, "data.csv")
    df.to_csv(csv_path, index=False)
    print(f"Wrote {n} clock images to '{out_dir}/' and metadata to '{csv_path}'")
    return df


# ---------------------------------------------------------------------------
# CLI
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Generate synthetic analog clock images.")
    parser.add_argument("--n", type=int, default=500, help="number of clocks to generate")
    parser.add_argument("--out_dir", type=str, default="data", help="output directory")
    parser.add_argument("--seed", type=int, default=42, help="random seed")
    args = parser.parse_args()

    generate_dataset(n=args.n, out_dir=args.out_dir, seed=args.seed)


In [ ]:
from clocks import generate_dataset

clock_df = generate_dataset(n=500, out_dir="data", seed=42)
clock_df.head()


Preview a few generated clocks:

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, (_, row) in zip(axes, clock_df.head(4).iterrows()):
    img = Image.open(f"data/{row['filename']}")
    ax.imshow(img)
    ax.set_title(f"{row['hour']}:{row['minute']:02d}")
    ax.axis("off")
plt.tight_layout()
plt.show()


## 2. Evaluation script (`eval_behavior.py`)

Loads Qwen2.5-VL-3B-Instruct in float16, prompts it with every clock image, parses the HH:MM answer, and logs parse failures separately.

In [ ]:
%%writefile eval_behavior.py
"""
eval_behavior.py -- STEP 1 behavior check: ask a VLM what time each clock shows.

Loads Qwen2.5-VL-3B-Instruct in float16, shows it each clock image generated
by clocks.py, asks it to read the time, and logs every answer (parsed and
raw) to a results CSV. This script does NOT do any interpretability -- it
only records model behavior for later analysis (analyze.py).

Usage as a script:
    python eval_behavior.py --data_csv data/data.csv --images_dir data \
        --out_csv results/results.csv

Usage as a library (e.g. from the notebook):
    from eval_behavior import run_eval
    results_df = run_eval(data_csv="data/data.csv", images_dir="data",
                           out_csv="results/results.csv")
"""

import argparse
import os
import re

import pandas as pd
import torch
from PIL import Image
from tqdm import tqdm
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, set_seed

PROMPT = "What time does this clock show? Answer only in HH:MM format."
MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"
SEED = 0

# Matches things like "3:45", "03:45", "12:05" anywhere in the model's reply.
TIME_RE = re.compile(r"(\d{1,2}):(\d{2})")


def parse_time_answer(text):
    """Extract (hour, minute) ints from a model reply string.

    Returns (hour, minute, success). success=False (hour/minute=None) if no
    HH:MM pattern was found, or if the numbers are out of a valid clock range.
    """
    match = TIME_RE.search(text)
    if not match:
        return None, None, False

    hour, minute = int(match.group(1)), int(match.group(2))

    # Sanity-check the range. Models sometimes output 24h-style hours (e.g.
    # "13:45") or garbage; treat those as parse failures too since our
    # ground truth is always a 12-hour analog reading (hour in 1-12).
    if not (0 <= hour <= 23) or not (0 <= minute <= 59):
        return None, None, False
    hour_12 = hour if 1 <= hour <= 12 else (hour - 12 if hour > 12 else 12)

    return hour_12, minute, True


def load_model(model_id=MODEL_ID):
    """Load the VLM + processor in float16. Uses device_map='auto' so it
    places the model on GPU if available (works on a Kaggle T4)."""
    print(f"Loading {model_id} in float16 ...")
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    processor = AutoProcessor.from_pretrained(model_id)
    model.eval()
    return model, processor


@torch.no_grad()
def ask_model(model, processor, image_path, prompt=PROMPT, max_new_tokens=16):
    """Send one clock image + prompt to the model, return its raw text reply."""
    image = Image.open(image_path).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    # Build the chat-formatted prompt text, then let the processor turn the
    # PIL image + text into model inputs (this handles image tokenization).
    chat_text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[chat_text], images=[image], return_tensors="pt").to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    # Strip the prompt tokens off the front so we only decode the new reply.
    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    reply = processor.batch_decode(trimmed, skip_special_tokens=True,
                                    clean_up_tokenization_spaces=False)[0]
    return reply.strip()


def run_eval(
    data_csv="data/data.csv",
    images_dir="data",
    out_csv="results/results.csv",
    model_id=MODEL_ID,
    max_new_tokens=16,
    max_images=None,
    seed=SEED,
):
    """Run the behavior check over every clock in `data_csv` and save results.

    Generation is deterministic: `do_sample=False` (greedy decoding) in
    `ask_model`, plus a fixed seed set here for full reproducibility (greedy
    decoding shouldn't need one, but this also pins any other randomness,
    e.g. in kernel selection). `max_new_tokens` is kept small since the
    expected answer is just "HH:MM".

    Writes:
      - out_csv: one row per image with the true time, the model's raw text
        reply, the parsed prediction, and a parse_success flag.
      - <out_csv dir>/parse_failures.csv: just the rows that failed to parse,
        for quick inspection.

    Returns the results DataFrame.
    """
    set_seed(seed)

    df = pd.read_csv(data_csv)
    if max_images is not None:
        df = df.head(max_images)

    model, processor = load_model(model_id)

    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Evaluating clocks"):
        image_path = os.path.join(images_dir, row["filename"])
        raw_reply = ask_model(model, processor, image_path, max_new_tokens=max_new_tokens)
        pred_hour, pred_minute, ok = parse_time_answer(raw_reply)

        rows.append({
            "filename": row["filename"],
            "true_hour": row["hour"],
            "true_minute": row["minute"],
            "raw_answer": raw_reply,
            "pred_hour": pred_hour,
            "pred_minute": pred_minute,
            "parse_success": ok,
        })

    results_df = pd.DataFrame(rows)

    out_dir = os.path.dirname(out_csv) or "."
    os.makedirs(out_dir, exist_ok=True)
    results_df.to_csv(out_csv, index=False)

    failures = results_df[~results_df["parse_success"]]
    failures_path = os.path.join(out_dir, "parse_failures.csv")
    failures.to_csv(failures_path, index=False)

    n_fail = len(failures)
    print(f"Done. {len(results_df)} images evaluated, {n_fail} parse failures "
          f"({n_fail / len(results_df):.1%}).")
    print(f"Results saved to '{out_csv}', parse failures to '{failures_path}'.")

    return results_df


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Evaluate a VLM's clock-reading behavior.")
    parser.add_argument("--data_csv", type=str, default="data/data.csv",
                         help="CSV of clock metadata produced by clocks.py")
    parser.add_argument("--images_dir", type=str, default="data",
                         help="directory containing the clock PNGs")
    parser.add_argument("--out_csv", type=str, default="results/results.csv",
                         help="where to write the results CSV")
    parser.add_argument("--model_id", type=str, default=MODEL_ID)
    parser.add_argument("--max_new_tokens", type=int, default=16)
    parser.add_argument("--max_images", type=int, default=None,
                         help="only evaluate the first N images (useful for a quick test run)")
    parser.add_argument("--seed", type=int, default=SEED, help="random seed for reproducibility")
    args = parser.parse_args()

    run_eval(
        data_csv=args.data_csv,
        images_dir=args.images_dir,
        out_csv=args.out_csv,
        model_id=args.model_id,
        max_new_tokens=args.max_new_tokens,
        max_images=args.max_images,
        seed=args.seed,
    )


Run the full evaluation. On a T4 this takes a while for 500 images (roughly on the order of tens of minutes) -- set `max_images` to a small number first if you just want to sanity-check that everything works end to end.

In [ ]:
from eval_behavior import run_eval

# Quick sanity check on a handful of images first (comment out once verified).
# results_df = run_eval(data_csv="data/data.csv", images_dir="data",
#                        out_csv="results/results.csv", max_images=8)

results_df = run_eval(data_csv="data/data.csv", images_dir="data", out_csv="results/results.csv")
results_df.head()


## 3. Analysis (`analyze.py`)

Reports exact accuracy, hour/minute accuracy, hand-swap rate, accuracy within +/-5 minutes, breakdowns by hour and by minute, and saves example images for each error type.

In [ ]:
%%writefile analyze.py
"""
analyze.py -- STEP 1 behavior check: analyze the VLM's clock-reading results.

Reads the results CSV produced by eval_behavior.py and reports:
  - exact accuracy (hour AND minute both correct)
  - hour accuracy and minute accuracy separately
  - hand-swap rate (see `is_hand_swap` below for the exact heuristic), reported
    both including and excluding "overlap" clocks -- see `hands_overlap` --
    where the hour and minute hands sit close enough together (e.g. 12:00,
    1:05, 2:11) that a swap and a correct answer look almost identical
  - accuracy within +/- 5 minutes (circular distance on a 12h face)
  - accuracy broken down by minute-bucket (near :00 vs elsewhere) and by hour
  - example images of each error type, saved as PNG montages

Usage as a script:
    python analyze.py --results_csv results/results.csv --images_dir data \
        --out_dir analysis_output

Usage as a library (e.g. from the notebook):
    from analyze import run_analysis
    summary = run_analysis(results_csv="results/results.csv", images_dir="data")
"""

import argparse
import os

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

from clocks import hand_angles

# Hands within this many degrees of each other are considered "overlapping":
# at that point a swapped reading and a correct reading point at nearly the
# same place on the face, so a "hand swap" there is much less meaningful
# than one where the hands are clearly apart.
OVERLAP_THRESHOLD_DEG = 15


# ---------------------------------------------------------------------------
# Metric helpers
# ---------------------------------------------------------------------------

def circular_minute_diff(true_hour, true_minute, pred_hour, pred_minute):
    """Absolute difference in minutes between two times on a 12-hour clock
    face, accounting for wraparound (e.g. 11:58 vs 12:02 is 4 minutes apart,
    not 718). Hours are normalized so 12 == 0.
    """
    t_true = (true_hour % 12) * 60 + true_minute
    t_pred = (pred_hour % 12) * 60 + pred_minute
    diff = abs(t_true - t_pred)
    return min(diff, 720 - diff)


def is_hand_swap(true_hour, true_minute, pred_hour, pred_minute):
    """Heuristic: did the model read the hour hand as the minute hand and
    vice versa?

    On an analog face, the hour hand's position also lines up with a minute
    tick (hour H sits at the same angle as minute H*5), and the minute
    hand's position lines up with an hour number (minute M sits at the same
    angle as hour round(M/5)). If the model's answer matches the time you'd
    get by reading the hands backwards this way, we call it a hand swap.

    This is an approximation, not a perfect ground truth -- e.g. for times
    near quarter-hours a swapped reading can coincide with other error
    types. It's intended to give a rough rate, not a guarantee.
    """
    swapped_hour = round(true_minute / 5) % 12
    if swapped_hour == 0:
        swapped_hour = 12
    swapped_minute = (true_hour % 12) * 5

    exact = (pred_hour == true_hour) and (pred_minute == true_minute)
    return (not exact) and (pred_hour == swapped_hour) and (pred_minute == swapped_minute)


def hands_overlap(hour, minute, threshold_deg=OVERLAP_THRESHOLD_DEG):
    """True if the hour and minute hands point within `threshold_deg` of each
    other on the true clock face (e.g. near 12:00, 1:05, 2:11, ...).

    Uses the exact same angle formulas as the renderer (`clocks.hand_angles`)
    so this lines up with what the image actually looks like.
    """
    hour_angle, minute_angle = hand_angles(hour, minute)
    diff = abs(hour_angle - minute_angle) % 360
    diff = min(diff, 360 - diff)
    return diff <= threshold_deg


def minute_bucket(minute):
    """Bucket a minute value into a 5-minute-wide label, e.g. 'near :00'."""
    bucket_start = (minute // 5) * 5
    if bucket_start == 0:
        return "near :00"
    return f":{bucket_start:02d}-:{bucket_start + 4:02d}"


def categorize_row(row):
    """Assign a single error-type label to a result row, in priority order."""
    if not row["parse_success"]:
        return "parse_failure"

    th, tm, ph, pm = row["true_hour"], row["true_minute"], row["pred_hour"], row["pred_minute"]
    ph, pm = int(ph), int(pm)

    if ph == th and pm == tm:
        return "correct"
    if is_hand_swap(th, tm, ph, pm):
        return "hand_swap"
    if ph == th:
        return "hour_only_correct"
    if pm == tm:
        return "minute_only_correct"
    if circular_minute_diff(th, tm, ph, pm) <= 5:
        return "within_5min"
    return "other_error"


# ---------------------------------------------------------------------------
# Main analysis
# ---------------------------------------------------------------------------

def run_analysis(results_csv="results/results.csv", images_dir="data",
                  out_dir="analysis_output", examples_per_category=6,
                  overlap_threshold_deg=OVERLAP_THRESHOLD_DEG):
    df = pd.read_csv(results_csv)
    os.makedirs(out_dir, exist_ok=True)

    n_total = len(df)
    parsed = df[df["parse_success"]].copy()
    n_parsed = len(parsed)

    # Types coming back from CSV are floats if there were any NaNs; make the
    # parsed subset's prediction columns plain ints for comparisons.
    parsed["pred_hour"] = parsed["pred_hour"].astype(int)
    parsed["pred_minute"] = parsed["pred_minute"].astype(int)

    parsed["error_type"] = parsed.apply(categorize_row, axis=1)
    parsed["minute_diff"] = parsed.apply(
        lambda r: circular_minute_diff(r["true_hour"], r["true_minute"], r["pred_hour"], r["pred_minute"]),
        axis=1,
    )
    parsed["is_overlap"] = parsed.apply(
        lambda r: hands_overlap(r["true_hour"], r["true_minute"], overlap_threshold_deg), axis=1
    )

    exact_correct = (parsed["error_type"] == "correct").sum()
    hour_correct = (parsed["pred_hour"] == parsed["true_hour"]).sum()
    minute_correct = (parsed["pred_minute"] == parsed["true_minute"]).sum()
    hand_swaps = (parsed["error_type"] == "hand_swap").sum()
    within_5min = (parsed["minute_diff"] <= 5).sum()

    # Hand-swap rate is easy to inflate near-overlap times (e.g. 1:05), where
    # a swapped reading and a correct one point almost the same place on the
    # face -- so report it both ways: over every parsed clock, and over only
    # the clocks where the hands are clearly apart.
    non_overlap = parsed[~parsed["is_overlap"]]
    n_overlap = int(parsed["is_overlap"].sum())
    n_non_overlap = len(non_overlap)
    hand_swaps_non_overlap = (non_overlap["error_type"] == "hand_swap").sum()

    summary = {
        "n_total_images": n_total,
        "n_parsed": n_parsed,
        "parse_failure_rate": 1 - n_parsed / n_total if n_total else float("nan"),
        "exact_accuracy": exact_correct / n_parsed if n_parsed else float("nan"),
        "hour_accuracy": hour_correct / n_parsed if n_parsed else float("nan"),
        "minute_accuracy": minute_correct / n_parsed if n_parsed else float("nan"),
        "within_5min_accuracy": within_5min / n_parsed if n_parsed else float("nan"),
        "n_overlap_clocks": n_overlap,
        "overlap_rate": n_overlap / n_parsed if n_parsed else float("nan"),
        "hand_swap_rate_including_overlap": hand_swaps / n_parsed if n_parsed else float("nan"),
        "hand_swap_rate_excluding_overlap": (
            hand_swaps_non_overlap / n_non_overlap if n_non_overlap else float("nan")
        ),
    }

    # --- breakdown by hour ---
    # Named aggregation (rather than groupby().apply()) works the same across
    # older and newer pandas versions, which matters since Kaggle's pinned
    # pandas version can lag behind.
    by_hour = parsed.groupby("true_hour").agg(
        n=("error_type", "size"),
        exact_accuracy=("error_type", lambda s: (s == "correct").mean()),
        hand_swap_rate=("error_type", lambda s: (s == "hand_swap").mean()),
    ).reset_index()
    by_hour.to_csv(os.path.join(out_dir, "by_hour.csv"), index=False)

    # --- breakdown by minute bucket (near :00 vs elsewhere) ---
    parsed["minute_bucket"] = parsed["true_minute"].apply(minute_bucket)
    by_minute = parsed.groupby("minute_bucket").agg(
        n=("error_type", "size"),
        exact_accuracy=("error_type", lambda s: (s == "correct").mean()),
        hand_swap_rate=("error_type", lambda s: (s == "hand_swap").mean()),
    ).reset_index()
    by_minute.to_csv(os.path.join(out_dir, "by_minute_bucket.csv"), index=False)

    # --- print summary ---
    print("=== Clock-reading behavior summary ===")
    for k, v in summary.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.1%}" if "rate" in k or "accuracy" in k else f"  {k}: {v}")
        else:
            print(f"  {k}: {v}")

    with open(os.path.join(out_dir, "summary.txt"), "w") as f:
        for k, v in summary.items():
            f.write(f"{k}: {v}\n")

    # --- example images per error type ---
    examples_dir = os.path.join(out_dir, "examples")
    os.makedirs(examples_dir, exist_ok=True)
    error_categories = ["correct", "hand_swap", "hour_only_correct",
                         "minute_only_correct", "within_5min", "other_error"]
    for category in error_categories:
        subset = parsed[parsed["error_type"] == category]
        if len(subset) == 0:
            continue
        save_example_grid(subset.head(examples_per_category), images_dir,
                           os.path.join(examples_dir, f"{category}.png"), category)

    # Parse failures don't have a usable prediction to caption, but are still
    # worth a quick look.
    parse_fail_subset = df[~df["parse_success"]].head(examples_per_category)
    if len(parse_fail_subset) > 0:
        save_example_grid(parse_fail_subset, images_dir,
                           os.path.join(examples_dir, "parse_failure.png"), "parse_failure")

    print(f"\nBreakdowns and example images written to '{out_dir}/'.")
    return summary


def save_example_grid(rows, images_dir, save_path, category):
    """Save a small montage of example clock images with true/predicted
    time captions, for quick visual inspection of one error category."""
    n = len(rows)
    ncols = min(3, n)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4.5 * nrows))
    axes = [axes] if n == 1 else axes.flatten()

    for ax, (_, row) in zip(axes, rows.iterrows()):
        img_path = os.path.join(images_dir, row["filename"])
        img = Image.open(img_path)
        ax.imshow(img)
        ax.axis("off")

        true_str = f"{int(row['true_hour'])}:{int(row['true_minute']):02d}"
        if row["parse_success"]:
            pred_str = f"{int(row['pred_hour'])}:{int(row['pred_minute']):02d}"
        else:
            pred_str = f"(unparsed: {row['raw_answer']!r})"
        ax.set_title(f"true={true_str}  pred={pred_str}", fontsize=10)

    # hide any unused axes
    for ax in axes[n:]:
        ax.axis("off")

    fig.suptitle(f"Error type: {category}", fontsize=13)
    fig.tight_layout()
    fig.savefig(save_path, dpi=100)
    plt.close(fig)


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Analyze VLM clock-reading results.")
    parser.add_argument("--results_csv", type=str, default="results/results.csv")
    parser.add_argument("--images_dir", type=str, default="data")
    parser.add_argument("--out_dir", type=str, default="analysis_output")
    parser.add_argument("--examples_per_category", type=int, default=6)
    parser.add_argument("--overlap_threshold_deg", type=float, default=OVERLAP_THRESHOLD_DEG,
                         help="hands within this many degrees of each other count as 'overlapping'")
    args = parser.parse_args()

    run_analysis(
        results_csv=args.results_csv,
        images_dir=args.images_dir,
        out_dir=args.out_dir,
        examples_per_category=args.examples_per_category,
        overlap_threshold_deg=args.overlap_threshold_deg,
    )


In [ ]:
from analyze import run_analysis

summary = run_analysis(results_csv="results/results.csv", images_dir="data",
                        out_dir="analysis_output")
summary


Breakdown tables:

In [ ]:
import pandas as pd

by_hour = pd.read_csv("analysis_output/by_hour.csv")
by_minute = pd.read_csv("analysis_output/by_minute_bucket.csv")
display(by_hour)
display(by_minute)


Example images for each error type (saved as montage PNGs under `analysis_output/examples/`):

In [ ]:
import glob

for path in sorted(glob.glob("analysis_output/examples/*.png")):
    print(path)
    display(Image.open(path))


## Next steps

This notebook only characterizes *whether and how* the model gets clock-reading wrong (Step 1). The next step in the project is interpretability work (e.g. attention/activation analysis) to understand *why* -- not included here by design.